# Binarization

**Binarization** converts a numeric column into just two values (0 or 1) based on a threshold — the simplest possible form of discretization. Here, the engineered `family` column (`SibSp + Parch`, i.e. total family members aboard) gets reduced to just "traveling alone" (0) vs. "traveling with family" (1).

## 1. Setup

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

## 2. Load Data and Engineer the `family` Feature

In [3]:
df = pd.read_csv('train.csv')[['Age','Fare','SibSp','Parch','Survived']]

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.head()

Combining `SibSp` (siblings/spouses aboard) and `Parch` (parents/children aboard) into one `family` count is simpler for a model to use directly than two separate small integer columns.

In [ ]:
df['family'] = df['SibSp'] + df['Parch']

In [ ]:
df.head()

In [ ]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [ ]:
df.head()

## 3. Train/Test Split

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

### Visualizing Why Binarization Might Help

Before touching the model, it's worth seeing whether survival actually differs between "alone" and "with family" — if it doesn't, binarizing won't help any model.

In [ ]:
import matplotlib.pyplot as plt

survival_by_alone = pd.concat([X_train['family'], y_train], axis=1)
survival_by_alone['alone'] = (survival_by_alone['family'] == 0)
survival_by_alone.groupby('alone')['Survived'].mean().plot(kind='bar', color=['#DD8452', '#4C72B0'])
plt.ylabel('Survival rate')
plt.title('Survival Rate: Alone vs. Traveling with Family')
plt.xticks([0, 1], ['With family', 'Alone'], rotation=0)
plt.show()


## 4. Baseline — No Binarization

In [ ]:
# Without binarization

clf = DecisionTreeClassifier()

clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)

accuracy_score(y_test,y_pred)

In [ ]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

## 5. Applying Binarization

`Binarizer(threshold=0.0)` (the default) maps any value `> 0` to `1`, and everything `≤ 0` to `0` — exactly "has any family aboard" vs. "traveling alone". `copy=False` modifies the array in place instead of allocating a new one, a minor memory-saving detail.

In [ ]:
# Applying Binarization

from sklearn.preprocessing import Binarizer

In [ ]:
trf = ColumnTransformer([
    ('bin',Binarizer(copy=False),['family'])
],remainder='passthrough')

In [ ]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [ ]:
pd.DataFrame(X_train_trf,columns=['family','Age','Fare'])

### Accuracy After Binarization

In [ ]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

### Cross-Validated Comparison

In [ ]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

## 6. Does Binarization Actually Help a Decision Tree?

Worth being honest about a subtlety here: a decision tree can already learn a threshold split like "`family > 0`" **on its own**, from the raw `family` column — trees don't need help finding thresholds the way linear models do. Binarization tends to matter far more for models like logistic regression, which can't easily discover the same non-linear cutoff by itself. Comparing both model types makes that difference visible.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)
acc_lr_raw = accuracy_score(y_test, lr.predict(X_test))

lr_bin = LogisticRegression()
lr_bin.fit(X_train_trf, y_train)
acc_lr_bin = accuracy_score(y_test, lr_bin.predict(X_test_trf))

print('Logistic Regression — raw family:        ', acc_lr_raw)
print('Logistic Regression — binarized family:  ', acc_lr_bin)


## Summary

| Task | Method |
|---|---|
| Turn a numeric column into 0/1 by threshold | `Binarizer(threshold=0.0)` |
| Apply to just one column | `ColumnTransformer([('bin', Binarizer(), ['col'])], remainder='passthrough')` |
| Reliable before/after comparison | `cross_val_score(model, X, y, cv=10)` |

### Who actually benefits from binarization
Trees can already split on a raw numeric threshold themselves — binarizing rarely changes their accuracy much. Linear models (logistic regression, SVMs) benefit more, since they otherwise can't easily represent a sharp non-linear cutoff like "family > 0" on their own.